# Module 9 lab · Information

The code from every lesson of [this module](https://aicourses.dev/mathematics/information), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/mathematics/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 9.2 Entropy

Entropy is the average surprise of a source, and it tells you the fewest questions you need on average. You compute it for a fair coin, a loaded coin and a table of next-token chances.

[Read the lesson](https://aicourses.dev/mathematics/information/entropy)

### Surprise of one outcome

For the loaded coin, heads has chance 0.9 and tails 0.1. Heads is $-\log_2 0.9 = 0.152$ bits and tails is $-\log_2 0.1 = 3.322$ bits. The code gives these:

In [ ]:
import numpy as np

for p in [1 / 2, 1 / 8, 0.9, 0.1]:
    print(p, round(-np.log2(p), 3))

*The lesson shows:*

```text
0.5 1.0
0.125 3.0
0.9 0.152
0.1 3.322
```

### Average surprise

Weighted: $0.40 \times 1.322 + 0.25 \times 2 + 0.15 \times 2.737 + 0.10 \times 3.322 + 0.10 \times 3.322 = 0.529 + 0.5 + 0.411 + 0.332 + 0.332 = 2.104$ bits. First as a loop, which adds one term at a time, then as one line of NumPy:

In [ ]:
def entropy(p):
    total = 0.0
    for pi in p:
        if pi > 0:
            total += pi * -np.log2(pi)
    return total

table = [0.40, 0.25, 0.15, 0.10, 0.10]
print(entropy([0.5, 0.5]), round(entropy([0.9, 0.1]), 3),
      round(entropy(table), 3))
p = np.array(table)
print(round(-(p * np.log2(p)).sum(), 3))

*The lesson shows:*

```text
1.0 0.469 2.104
2.104
```

### A chance of zero

What if an outcome has chance 0? Its surprise, $-\log_2 0$, is infinite, and a program that multiplies zero by infinity returns `nan`. The convention is that **the term for a chance of zero is taken as zero**. That is not a trick: as a chance $p$ shrinks towards 0, $p \times (-\log_2 p)$ shrinks towards 0 as well, because the chance falls faster than the surprise grows. Look at it in numbers, and then at the code that leaves such terms out:

In [ ]:
for c in [0.1, 0.01, 1e-6, 1e-12]:
    print(c, "%.3g" % (c * -np.log2(c)))
q = np.array([0.5, 0.5, 0.0])
with np.errstate(divide="ignore", invalid="ignore"):
    print(q * -np.log2(q))
print((-(q[q > 0] * np.log2(q[q > 0]))).sum())

*The lesson shows:*

```text
0.1 0.332
0.01 0.0664
1e-06 1.99e-05
1e-12 3.99e-11
0.5
[0.5 0.5 nan]
1.0
```

### Bits and nats

For the five-token table, the entropy in nats is $0.40 \times 0.916 + 0.25 \times 1.386 + 0.15 \times 1.897 + 0.10 \times 2.303 + 0.10 \times 2.303 = 0.366 + 0.347 + 0.285 + 0.230 + 0.230 = 1.458$, and $1.458 / 0.6931 = 2.104$, the number of bits found before.

In [ ]:
nats = -(p * np.log(p)).sum()
print(round(nats, 4), round(nats / np.log(2), 4), round(np.log2(5), 4))

*The lesson shows:*

```text
1.4582 2.1037 2.3219
```

## 9.3 Softmax and temperature

Softmax turns any list of scores into chances, and dividing the scores first, a temperature, makes it sharper or flatter. It is the last step of every classifier and language model.

[Read the lesson](https://aicourses.dev/mathematics/information/softmax-and-temperature)

### From scores to chances

where $\sum_j e^{s_j}$ means add up $e^{s_j}$ over every token $j$, and $q$ is the table the model produces. This is the **softmax** of the scores. By hand, for 2, 1 and 0: $e^2 = 7.389$, $e^1 = 2.718$ and $e^0 = 1$, which add to 11.107. The chances are $7.389 / 11.107 = 0.665$, $2.718 / 11.107 = 0.245$ and $1 / 11.107 = 0.090$. They add to 1, the top score gets the biggest share, and the score 0 gets 0.09, not 0. First as a loop that follows the two moves, then as a function:

In [ ]:
import numpy as np

s = [2.0, 1.0, 0.0]
e = [float(np.exp(x)) for x in s]
total = sum(e)
print([round(x, 3) for x in e], round(total, 3))
print([round(x / total, 4) for x in e])

*The lesson shows:*

```text
[7.389, 2.718, 1.0] 11.107
[0.6652, 0.2447, 0.09]
```

In [ ]:
def softmax(s, T=1.0):
    e = np.exp(np.array(s) / T)
    return e / e.sum()

print(softmax([2, 1, 0]).round(4))

*The lesson shows:*

```text
[0.6652 0.2447 0.09  ]
```

### Shifting every score

Add 10 to every score: 12, 11 and 10. Does anything change? Predict, then look. The factor for each token is $e^{12} = e^{10} \times e^2$, $e^{11} = e^{10} \times e^1$ and $e^{10} = e^{10} \times e^0$ (adding exponents multiplies powers). So every top number and the total share the same factor $e^{10}$, and it cancels when you divide. The chances are those of 2, 1 and 0 exactly. The same holds for subtracting 10:

In [ ]:
print(softmax([12, 11, 10]).round(4))
print(softmax([-8, -9, -10]).round(4))

*The lesson shows:*

```text
[0.6652 0.2447 0.09  ]
[0.6652 0.2447 0.09  ]
```

In [ ]:
print(softmax([3, 1, 0]).round(4), softmax([1, -1, -2]).round(4))

*The lesson shows:*

```text
[0.8438 0.1142 0.042 ] [0.8438 0.1142 0.042 ]
```

### Temperature

The code adds the entropy in bits of each table (lesson 2), because sharper means less average surprise:

In [ ]:
for T in [0.5, 1, 2]:
    q = softmax([2, 1, 0], T)
    print(T, q.round(3), round(-(q * np.log2(q)).sum(), 3))

*The lesson shows:*

```text
0.5 [0.867 0.117 0.016] 0.636
1 [0.665 0.245 0.09 ] 1.201
2 [0.506 0.307 0.186] 1.472
```

## 9.4 Cross-entropy: the loss language models train on

Cross-entropy is the loss language models train on: the surprise at what really came next, averaged. It is the likelihood of Module 08 with its sign flipped.

[Read the lesson](https://aicourses.dev/mathematics/information/cross-entropy)

### The loss for one example

The **cross-entropy** is the average of these losses over the examples. With ten examples, six of 0.693, two of 1.386, one of 1.897 and one of 2.303, it is $(4.159 + 2.773 + 1.897 + 2.303) / 10 = 1.113$ nats. In code, a loop that follows the sentence, then one line:

In [ ]:
import numpy as np

q = np.array([0.50, 0.25, 0.15, 0.10])          # cat, dog, mat, hat
x = np.array([0, 0, 0, 0, 0, 0, 1, 1, 2, 3])     # the ten real next words
losses = [-np.log(q[k]) for k in x]
print(np.round(losses, 3))
print(round(np.mean(losses), 4), round(-np.log(q[x]).mean(), 4))

*The lesson shows:*

```text
[0.693 0.693 0.693 0.693 0.693 0.693 1.386 1.386 1.897 2.303]
1.1131 1.1131
```

### The likelihood with its sign flipped

The second form uses the rule that the log of a product is the sum of the logs, and $\prod$ means multiply over every example. Read it as: the cross-entropy is minus the log-likelihood, divided by $N$. With the ten words, the log of the product is $-11.131$, and $11.131 / 10 = 1.113$.

In [ ]:
lik = np.prod(q[x])
print("%.4g" % lik, round(np.log(lik), 3), round(-np.log(lik) / len(x), 4))

*The lesson shows:*

```text
1.465e-05 -11.131 1.1131
```

### Grouped by word

This is the **cross-entropy** of the model’s table $q$ against the data’s table $p$, which is where the name comes from: the entropy formula of lesson 2, with the model’s table inside the log and the data’s table doing the weighting. By hand: $0.6 \times 0.693 + 0.2 \times 1.386 + 0.1 \times 1.897 + 0.1 \times 2.303 = 0.416 + 0.277 + 0.190 + 0.230 = 1.113$ nats. The same number as the ten separate losses. When there is one example, $p$ is a table with a 1 at the word that came and 0 elsewhere, and the formula collapses to $-\ln q(\text{that word})$, which is the loss for one example.

In [ ]:
p = np.bincount(x) / len(x)
H_pq = -(p * np.log(q)).sum()
print(p, round(H_pq, 4), round(H_pq / np.log(2), 3))

*The lesson shows:*

```text
[0.6 0.2 0.1 0.1] 1.1131 1.606
```

In [ ]:
print(round(-np.log([0.5, 0.25, 0.125, 0.5]).mean(), 3))

*The lesson shows:*

```text
1.213
```

## 9.5 KL divergence

KL divergence is the extra surprise you pay for using the wrong table, and it is not a distance because it differs each way round. You see it as the gap between cross-entropy and entropy.

[Read the lesson](https://aicourses.dev/mathematics/information/kl-divergence)

### The extra surprise

read “the divergence from $p$ to $q$”. The second form follows from the first. The cross-entropy is $-\sum_k p_k \ln q_k$ and the entropy, in nats, is $-\sum_k p_k \ln p_k$. Subtracting the second from the first gives $\sum_k p_k (\ln p_k - \ln q_k)$, and $\ln p_k - \ln q_k = \ln \frac{p_k}{q_k}$. By hand: $0.6 \ln 1.2 + 0.2 \ln 0.8 + 0.1 \ln \frac{2}{3} + 0.1 \ln 1 = 0.1094 - 0.0446 - 0.0405 + 0 = 0.0243$ (with the terms rounded as shown; the unrounded sum is 0.0242, the number the code prints). The model costs about 0.024 nats per word more than the best table.

In [ ]:
import numpy as np

p = np.array([0.6, 0.2, 0.1, 0.1])        # the data’s table
q = np.array([0.50, 0.25, 0.15, 0.10])    # the model’s table
H_p = -(p * np.log(p)).sum()
H_pq = -(p * np.log(q)).sum()
print(round(H_p, 4), round(H_pq, 4), round(H_pq - H_p, 4))
print(round((p * np.log(p / q)).sum(), 4))

*The lesson shows:*

```text
1.0889 1.1131 0.0242
0.0242
```

### Not a distance

From $a$ to $u$: $D_{\mathrm{KL}}(a \,\|\, u) = \ln 3 - H(a)$, since the cross-entropy with a flat table is always $\ln 3$ (every term is $a_k \ln 3$, and the $a_k$ add to 1). $H(a) = 0.9 \times 0.1054 + 2 \times 0.05 \times 2.9957 = 0.3944$, so the divergence is $1.0986 - 0.3944 = 0.7042$ nats. From $u$ to $a$: $\frac13 \left( \ln \frac{1/3}{0.9} + 2 \ln \frac{1/3}{0.05} \right) = \frac13 (-0.9933 + 2 \times 1.8971) = 0.9337$ nats.

In [ ]:
kl = lambda p, q: (p * np.log(p / q)).sum()
a = np.array([0.9, 0.05, 0.05])
u = np.ones(3) / 3
print(round(kl(a, u), 4), round(kl(u, a), 4))
print(round(kl(a, u) / np.log(2), 3), round(kl(u, a) / np.log(2), 3))

*The lesson shows:*

```text
0.7042 0.9337
1.016 1.347
```

### Never negative, zero only when equal

Two facts hold for every pair of tables. The divergence is never below 0, and it is exactly 0 only when the two tables are the same. Both can be proved, with a short argument about the curve of $\ln x$ that this lesson does not give; here you check them in numbers. The code draws 100,000 random pairs of five-outcome tables with a seeded generator and computes the divergence of each, and then compares a table with itself.

In [ ]:
rng = np.random.default_rng(0)
P = rng.dirichlet(np.ones(5), size=100_000)
Q = rng.dirichlet(np.ones(5), size=100_000)
vals = (P * np.log(P / Q)).sum(axis=1)
print(vals.min() > 0, round(vals.min(), 6), round(vals.mean(), 3))
print((P[0] * np.log(P[0] / P[0])).sum())

*The lesson shows:*

```text
True 0.001826 0.802
0.0
```

## 9.6 Perplexity

Perplexity turns cross-entropy back into how many equally likely choices the model is torn between. It is the number most language-model reports use.

[Read the lesson](https://aicourses.dev/mathematics/information/perplexity)

### From loss to a count

By hand, for the four-word sentence. The chances are 0.5, 0.25, 0.125 and 0.25. The surprises in bits are 1, 2, 3 and 2, which average 2 bits, and $2^2 = 4$. In nats they are 0.693, 1.386, 2.079 and 1.386, which average 1.386, and $e^{1.386} = 4$. A third route, from the chances themselves: the chance of the whole sentence is the product, $0.5 \times 0.25 \times 0.125 \times 0.25 = 1/256$, and the perplexity is that chance raised to the power $-1/4$ (minus one over the number of words), the same as the fourth root of 256, which is 4. This is the form in Jurafsky and Martin’s equations: the inverse of the sentence’s chance, scaled for its length, a geometric average of the model’s inverse chances.

In [ ]:
import numpy as np

c = [0.5, 0.25, 0.125, 0.25]     # chances a model gave four real words
prod = np.prod(c)
print(prod, prod ** (-1 / 4))
bits = -np.log2(c)
print(bits, bits.mean(), 2 ** bits.mean())
nats = -np.log(c)
print(round(nats.mean(), 4), round(np.exp(nats.mean()), 4))

*The lesson shows:*

```text
0.00390625 4.0
[1. 2. 3. 2.] 2.0 4.0
1.3863 4.0
```

Now the ten words of lesson 4, where the model’s table was 0.50, 0.25, 0.15 and 0.10 and the loss was 1.1131 nats:

In [ ]:
q = np.array([0.50, 0.25, 0.15, 0.10])
x = np.array([0, 0, 0, 0, 0, 0, 1, 1, 2, 3])
L = -np.log(q[x]).mean()
print(round(L, 4), round(np.exp(L), 3), round(2 ** (L / np.log(2)), 3))

*The lesson shows:*

```text
1.1131 3.044 3.044
```

In [ ]:
print(2 ** 3.0, round(np.exp(0.6931), 3), round(np.exp(np.log(46)), 1))

*The lesson shows:*

```text
8.0 2.0 46.0
```

## 9.7 Numbers that break: log-sum-exp

Raise a score high enough and softmax returns nan. Subtracting the largest score first fixes it, and the trick uses the logarithms and floating-point limits of Module 01.

[Read the lesson](https://aicourses.dev/mathematics/information/log-sum-exp)

### Breaking softmax

The exponential overflows when $e^s$ is larger than the largest float the format can hold. The largest 32-bit float is about $3.4028 \times 10^{38}$ and the largest 64-bit float about $1.7977 \times 10^{308}$, so the score where $e^s$ passes them is the natural log of those numbers. Compute them, then run the plain softmax just below and just above each:

In [ ]:
import numpy as np

print(np.log(np.finfo(np.float32).max), np.log(np.finfo(np.float64).max))
def naive(s):
    e = np.exp(s)
    return e / e.sum()
with np.errstate(all="ignore"):
    for dt, big in [(np.float32, 88), (np.float32, 89),
                    (np.float64, 709), (np.float64, 710)]:
        print(dt.__name__, big, naive(np.array([big, 0, 0], dtype=dt)))

*The lesson shows:*

```text
88.72284 709.782712893384
float32 88 [1.000000e+00 6.054601e-39 6.054601e-39]
float32 89 [nan  0.  0.]
float64 709 [1.00000000e+000 1.21678075e-308 1.21678075e-308]
float64 710 [nan  0.  0.]
```

At 88 in float32 and 709 in float64 the plain softmax still works. One step more and the first entry is `nan`. A scan in steps of 0.01 finds the edges themselves:

In [ ]:
def edge(dt, lo, hi):
    ok = None
    for s in np.arange(lo, hi, 0.01).astype(dt):
        if not np.isfinite(np.exp(s)):
            return ok, s
        ok = s
with np.errstate(all="ignore"):
    print([round(float(v), 2) for v in edge(np.float32, 88, 89)])
    print([round(float(v), 2) for v in edge(np.float64, 709, 710)])

*The lesson shows:*

```text
[88.72, 88.73]
[709.78, 709.79]
```

### Subtract the largest

The second form is the first with the top and the bottom both multiplied by $e^{-m}$, which is allowed because it is the same factor in both, and $e^{s_i} \times e^{-m} = e^{s_i - m}$. Now the largest shifted score is exactly 0, so the biggest exponential is $e^0 = 1$ and nothing can overflow. The total is at least 1, so it cannot be 0 either. Other terms may become so small that the format cannot hold them and stores 0 (this is called **underflow**), but a share that tiny is 0 for all practical purposes.

In [ ]:
def stable(s):
    e = np.exp(s - s.max())
    return e / e.sum()

s = np.array([2.0, 1.0, 0.0])
print(naive(s).round(4), stable(s).round(4))
print(stable(np.array([1000.0, 0.0, 0.0])).round(4))
print(stable(np.array([89, 0, 0], dtype=np.float32)).round(4))

*The lesson shows:*

```text
[0.6652 0.2447 0.09  ] [0.6652 0.2447 0.09  ]
[1. 0. 0.]
[1. 0. 0.]
```

### The log of a softmax

Cross-entropy needs the log of a chance: $\ln q_k$, for the true token $k$. The obvious way is to compute the softmax and then take its log, and it has a second problem that subtracting the largest does not cure. Take scores 200, 0 and 0 in 32-bit floats, with the true token the second one, the one the model thought very unlikely. The correct chance is about $e^{-200}$, and the correct loss is about 200 nats. The plain softmax is `nan` for the first entry; the stable one gives a 0 where the small chance should be, because $e^{-200}$ is smaller than the smallest float32, so it underflows to 0, and the log of 0 is minus infinity:

In [ ]:
s = np.array([200, 0, 0], dtype=np.float32)
with np.errstate(all="ignore"):
    print(np.log(naive(s)))
    print(np.log(stable(s)))

*The lesson shows:*

```text
[ nan -inf -inf]
[  0. -inf -inf]
```

because $e^{s_j} = e^{m} e^{s_j - m}$, and the log of $e^m$ times a sum is $m$ plus the log of the sum. By hand, for the scores 1000, 1001 and 1002: take $m = 1002$; the shifted scores are $-2$, $-1$ and 0, whose exponentials are 0.1353, 0.3679 and 1, adding to 1.5032, whose log is 0.4076, so the log-sum-exp is $1002.4076$. Each part of that is an ordinary number. For the score 200 case:

In [ ]:
lse = s.max() + np.log(np.exp(s - s.max()).sum())
print(lse, s - lse)

*The lesson shows:*

```text
200.0 [   0. -200. -200.]
```

In [ ]:
from scipy.special import logsumexp

print(logsumexp(np.array([500, 500], dtype=np.float32)))
print(logsumexp([1000, 1001, 1002]))

*The lesson shows:*

```text
500.69315
1002.4076059644444
```

## 9.8 The gradient of softmax with cross-entropy

Softmax with cross-entropy has a gradient that comes out as the chances minus the right answer. It is the gradient that starts backpropagation in Module 12.

[Read the lesson](https://aicourses.dev/mathematics/information/softmax-gradient)

### The three slopes

Now check it, as lesson 9 of Module 04 taught, with a central difference on each score, step $10^{-5}$:

In [ ]:
import numpy as np

def softmax(Z):
    E = np.exp(Z - Z.max(axis=-1, keepdims=True))
    return E / E.sum(axis=-1, keepdims=True)

s = np.array([2.0, 1.0, 0.0])
t = np.array([0.0, 1.0, 0.0])                 # the true class is the second
loss = lambda s: -np.log(softmax(s)[1])
h = 1e-5
num = [(loss(s + h * e) - loss(s - h * e)) / (2 * h) for e in np.eye(3)]
print(np.round(num, 4), (softmax(s) - t).round(4))

*The lesson shows:*

```text
[ 0.6652 -0.7553  0.09  ] [ 0.6652 -0.7553  0.09  ]
```

### From scores to weights

The division by $N$ is because the loss is the average over the batch. Here is the data, split as in Module 03 (1,297 to train on, 500 held out), with the loss and a check of one gradient entry against a central difference:

In [ ]:
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16
order = rng.permutation(len(X))
X, y = X[order], y[order]
Xtr, ytr, Xte, yte = X[:1297], y[:1297], X[1297:], y[1297:]
T = np.eye(10)[ytr]
ce = lambda W, b, X, y: -np.log(
    softmax(X @ W.T + b)[np.arange(len(y)), y]).mean()
W, b = np.zeros((10, 64)), np.zeros(10)
print(round(ce(W, b, Xtr, ytr), 4), round(np.log(10), 4))

*The lesson shows:*

```text
2.3026 2.3026
```

With all weights zero every class gets the chance 0.1, so the loss is $\ln 10 = 2.3026$, the “knows nothing” value of lesson 4. Now a gradient check at random weights, on three entries:

In [ ]:
W = rng.normal(0, 0.1, (10, 64))
S = (softmax(Xtr @ W.T + b) - T) / len(Xtr)
dW = S.T @ Xtr
for i, j in [(0, 5), (3, 20), (9, 63)]:
    E = np.zeros((10, 64)); E[i, j] = 1e-5
    num = (ce(W + E, b, Xtr, ytr) - ce(W - E, b, Xtr, ytr)) / 2e-5
    print(i, j, round(dW[i, j], 6), round(num, 6))

*The lesson shows:*

```text
0 5 0.01962 0.01962
3 20 -0.053006 -0.053006
9 63 0.000787 0.000787
```

Now use it: start from zeros and take 2,000 steps with learning rate 0.5.

In [ ]:
def train(lr, steps=2000):
    W, b = np.zeros((10, 64)), np.zeros(10)
    for step in range(1, steps + 1):
        S = (softmax(Xtr @ W.T + b) - T) / len(Xtr)
        W, b = W - lr * S.T @ Xtr, b - lr * S.sum(axis=0)
        if step in (10, 100, 500, 2000) and lr == 0.5:
            print(step, round(ce(W, b, Xtr, ytr), 4))
    return W, b

acc = lambda W, b, X, y: ((X @ W.T + b).argmax(axis=1) == y).mean()
for lr in (0.1, 0.5, 1.0):
    W, b = train(lr)
    print(lr, acc(W, b, Xte, yte))

*The lesson shows:*

```text
10 1.5403
100 0.4127
500 0.1768
2000 0.0905
0.1 0.962
0.5 0.982
1.0 0.98
```

## 9.9 Sampling: temperature, top-k and top-p

Temperature, top-k and top-p are all edits to the table before you draw. You see what each cuts, in numbers, so you can choose them on purpose.

[Read the lesson](https://aicourses.dev/mathematics/information/sampling)

### Temperature on a table

By hand with $T = 0.5$: the power is 2, so the chances are squared, $0.25$, $0.0625$, $0.0225$, $0.0049$ and $0.0009$, adding to $0.3408$, and renormalised they are $0.734$, $0.183$, $0.066$, $0.014$ and $0.003$. With $T = 2$ the power is $\frac12$, the square roots $0.707$, $0.5$, $0.387$, $0.265$ and $0.173$ add to $2.032$, giving $0.348$, $0.246$, $0.191$, $0.130$ and $0.085$.

In [ ]:
import numpy as np

tokens = ["cat", "dog", "bird", "fish", "emu"]
q = np.array([0.50, 0.25, 0.15, 0.07, 0.03])      # made up, most likely first

def temperature(q, T):
    w = q ** (1 / T)
    return w / w.sum()

def softmax(s):
    e = np.exp(s - s.max())
    return e / e.sum()

for T in (0.5, 2):
    print(T, temperature(q, T).round(3), softmax(np.log(q) / T).round(3))

*The lesson shows:*

```text
0.5 [0.734 0.183 0.066 0.014 0.003] [0.734 0.183 0.066 0.014 0.003]
2 [0.348 0.246 0.191 0.13  0.085] [0.348 0.246 0.191 0.13  0.085]
```

### Top-p and the nucleus

**Top-p**, also called **nucleus sampling**, is the edit Holtzman and co-authors proposed. Sort the tokens from most to least likely, and keep the smallest set whose chances add up to at least $p$ (their equation 2). Here $p$ is a threshold, a number such as 0.9, and not the data’s table of the earlier lessons. The kept set is the **nucleus**. With $p = 0.9$: the running total is 0.50, 0.75, 0.90. It reaches 0.9 with the third token, so three are kept, and renormalising gives the 0.556, 0.278 and 0.167 of the guess. The code for both edits:

In [ ]:
def top_k(q, k):
    out = np.zeros_like(q)
    keep = np.argsort(-q)[:k]
    out[keep] = q[keep]
    return out / out.sum()

def top_p(q, p):
    order = np.argsort(-q)
    n = np.searchsorted(np.cumsum(q[order]), p) + 1     # how many to keep
    out = np.zeros_like(q)
    out[order[:n]] = q[order[:n]]
    return out / out.sum()

print(top_k(q, 2).round(3), top_p(q, 0.9).round(3))

*The lesson shows:*

```text
[0.667 0.333 0.    0.    0.   ] [0.556 0.278 0.167 0.    0.   ]
```

In [ ]:
t = np.array([0.4, 0.3, 0.2, 0.1])
print(top_k(t, 2).round(3), top_p(t, 0.8).round(3))

*The lesson shows:*

```text
[0.571 0.429 0.    0.   ] [0.444 0.333 0.222 0.   ]
```

### A fixed k and a moving nucleus

Why bother with a second rule when top-k already cuts the tail? Because $k$ is the same for every table, and tables differ. Holtzman and co-authors argue that in some contexts the model’s chance is spread flat over tens of options and in others piled on one or two, so a small $k$ throws away good options in the first and a large one lets in bad options in the second, while a nucleus grows and shrinks with the table. Two tables show it:

In [ ]:
peaked = np.array([0.92, 0.04, 0.02, 0.01, 0.01])
flat = np.array([0.22, 0.21, 0.20, 0.19, 0.18])
for name, t in [("peaked", peaked), ("flat", flat)]:
    print(name, (top_k(t, 2) > 0).sum(), (top_p(t, 0.9) > 0).sum(),
          t[:2].sum().round(2))

*The lesson shows:*

```text
peaked 2 1 0.96
flat 2 5 0.43
```

### Drawing from the edited table

The draw is Module 07’s: lay the chances end to end, pick a uniform number, and take the stretch it lands in. The code below tallies 10,000 draws from the plain table and from three edited ones, with a seeded generator, so you can see which tokens survive:

In [ ]:
rng = np.random.default_rng(0)

def draw(table, n=10_000):
    cum = np.cumsum(table)
    cum[-1] = 1.0
    picks = np.searchsorted(cum, rng.random(n), side="right")
    return np.bincount(picks, minlength=len(table)) / n

for name, t in [("plain", q), ("T=0.5", temperature(q, 0.5)),
                ("top-k 2", top_k(q, 2)), ("top-p 0.9", top_p(q, 0.9))]:
    print(name, draw(t))

*The lesson shows:*

```text
plain [0.499  0.2505 0.1525 0.0681 0.0299]
T=0.5 [0.7292 0.1867 0.0657 0.016  0.0024]
top-k 2 [0.6704 0.3296 0.     0.     0.    ]
top-p 0.9 [0.5507 0.2855 0.1638 0.     0.    ]
```

## 9.10 Project: a next-character model made of counts

You build a next-character model from counts, score it with cross-entropy and perplexity on held-out text, and sample from it at three temperatures. It is the module’s ideas in one small language model.

[Read the lesson](https://aicourses.dev/mathematics/information/project-bigram-model)

**This is the project.** These cells are starting points from the lesson; the steps between them are yours to write. A cell that uses a name from one of your steps stops with a NameError until you have written that step.

### Step 1: load, clean and split

Lower-case the text and make every run of white space one space. That makes “Alice” and “alice” one character, and removes the line breaks of the printed page. Then split: the **first 90%** is for counting, and the **last 10%** is held out for scoring. The numbers should be 142,593 characters in all, 46 different ones, 128,333 for counting and 14,260 held out.

In [ ]:
import urllib.request
import numpy as np

rng = np.random.default_rng(0)
DATA = "https://raw.githubusercontent.com/aicoursesdev/mathematics-for-ai/main/data/"
urllib.request.urlretrieve(DATA + "alice.txt", "alice.txt")
text = " ".join(open("alice.txt", encoding="utf-8").read().lower().split())
chars = sorted(set(text))
V = len(chars)
idx = np.array([chars.index(c) for c in text])
cut = int(0.9 * len(idx))
train, test = idx[:cut], idx[cut:]
print(len(text), V, len(train), len(test))
print("".join(chars))

*The lesson shows:*

```text
142593 46 128333 14260
 !()*,-.:;?[]_abcdefghijklmnopqrstuvwxyzù—‘’“”
```

### Step 2: count and smooth

For every pair of neighbours in the training part, add 1 to the cell in row “the first character” and column “the one after it”. Start every cell at 1, not 0 (**add-one smoothing**): a pair never seen in training then still has a small chance, and does not give the model an infinite surprise when it appears in the held-out text (lesson 5). Each row of the counts, divided by its total, is the table of chances for the next character. A loop first, then the NumPy line that does it all at once:

In [ ]:
counts = np.ones((V, V))
for a, b in zip(train[:-1], train[1:]):
    counts[a, b] += 1
P = counts / counts.sum(axis=1, keepdims=True)
row = P[chars.index("q")]
print(P.shape, repr(chars[row.argmax()]), row.max().round(3))
fast = np.ones((V, V))
np.add.at(fast, (train[:-1], train[1:]), 1)
print((fast == counts).all(), P.sum(axis=1)[:3])

*The lesson shows:*

```text
(46, 46) 'u' 0.802
True [1. 1. 1.]
```

### Step 4: score it on held-out text

For each pair of neighbours in the held-out part, take $-\ln$ of the chance each model gave to the second character, and average (lesson 4). That is 14,259 pairs. Convert to bits by dividing by $\ln 2$, and to perplexity with the exponential (lesson 6).

In [ ]:
freq = np.bincount(train, minlength=V) + 1.0
freq /= freq.sum()
scores = {"uniform": np.log(V),
          "frequencies": -np.log(freq[test[1:]]).mean(),
          "bigram": -np.log(P[test[:-1], test[1:]]).mean()}
for name, L in scores.items():
    print("%-12s %.4f nats %.3f bits perplexity %.2f"
          % (name, L, L / np.log(2), np.exp(L)))

*The lesson shows:*

```text
uniform      3.8286 nats 5.524 bits perplexity 46.00
frequencies  3.0143 nats 4.349 bits perplexity 20.37
bigram       2.3465 nats 3.385 bits perplexity 10.45
```

### Step 5: sample at three temperatures

Draw characters one at a time from the model, starting at “a”: take the row of the last character, apply a temperature (lesson 9), and draw. One seeded generator, used for the three in turn, makes your samples match these.

In [ ]:
def sample(T, n=60, start="a"):
    out = [chars.index(start)]
    for _ in range(n - 1):
        w = P[out[-1]] ** (1 / T)
        out.append(rng.choice(V, p=w / w.sum()))
    return "".join(chars[i] for i in out)

for T in (0.5, 1.0, 1.5):
    print(T, repr(sample(T)))

*The lesson shows:*

```text
0.5 'ar a the sth t t the aine myouse as hers t t mof ather ilit '
1.0 'a id qur y arxithelyserediniry ttu uz—ayoun.—” ont th t tho '
1.5 'at heye. wabelithalory, ps it tho i_pepu[z*:(i’taitthe’latqu'
```